# Форма зеркала: собрать все лучи в одну точку

**Задача.** На зеркало сверху падает пучок параллельных лучей. Нужно, чтобы **все** они после отражения прошли через одну точку. Какой формы должно быть зеркало?

**Как запускать.** «Среда выполнения» → «Выполнить всё» (Ctrl+F9), затем крутить ползунки под графиком. Устанавливать ничего не нужно.

---

### Сценарий: нажимать по порядку

1. **«Сфера (k = 0)»** — первое, что приходит в голову. Край пучка собирается на высоте **0,750** вместо 1,000: промах **25 % от фокусного**. Справа видно то же самое числом: чем выше луч, тем ближе к зеркалу он пересекает ось.

2. Не меняя формы, **сузьте апертуру** ползунком до 0,15. Промах падает до **0,28 %**. Продольная аберрация растёт как $h^2$, поперечная — как $h^3$: сузили пучок вдвое — промах упал вчетверо. Поэтому фокус и называется **параксиальным**: сфера безупречна в пределе узкого пучка, а «парабола лучше сферы» — утверждение про широкие пучки.

3. **«Парабола (k = −1)»** — правая панель становится строго горизонтальной прямой. Подобрали.

4. Но подбор — не решение. Переключите задачу на **«вывести форму»**: там ничего не подбирается. Уравнение интегрируется от вершины наружу, шаг за шагом из условия отражения, и форма получается сама. Оранжевые кружки — коника $k=-1$ для сверки: расхождение порядка $10^{-10}$.

---

### Откуда берётся уравнение

Пусть $Y = y - f$ — высота, отсчитанная **от фокуса**. Условие «луч после отражения попал в фокус» в точке $(x, Y)$ даёт квадратное уравнение на наклон:

$$x\,y'^2 - 2Y\,y' - x = 0 \qquad\Longrightarrow\qquad y' = \frac{Y + \sqrt{x^2+Y^2}}{x} = \frac{x}{\sqrt{x^2+Y^2} - Y}$$

Второй вид — тот же после домножения на сопряжённое, и он лучше: в первом под вершиной вычитаются почти равные числа, а на самой оси получается $0/0$. Во втором ничего этого нет — **особенность на оси оказывается мнимой**, и интегрировать можно прямо от вершины.

**Уравнение однородное** — правая часть зависит только от отношения $Y/x$. И это можно было сказать **до всяких выкладок**: в условии «собрать все вертикальные лучи в одну точку» не участвует ни одна длина. Значит растяжение $(x, Y) \to (\lambda x, \lambda Y)$ обязано переводить решения в решения, а уравнение — зависеть только от безразмерного отношения.

Дальше стандартный ход. Подстановка $Y = v x$:

$$x\,v' = \sqrt{1+v^2} \quad\Longrightarrow\quad \operatorname{arsh} v = \ln x + C \quad\Longrightarrow\quad Y = \frac{A}{2}x^2 - \frac{1}{2A}$$

Парабола. Постоянная $A$ — это фокусное расстояние: **масштаб сидит не в уравнении, а в начальном условии**. Логарифм в ответе — подпись того же самого: раз своей длины у задачи нет, входить в ответ могут только отношения.

---

### Сверка с конспектом

На практике оси выбраны иначе: **фокус** в начале координат, вершина — в точке $V$ на оси, лучи идут вдоль $x$. Уравнение там выглядит так:

$$y' = \frac{y}{\sqrt{x^2+y^2}+x}, \qquad y^2 = 4V(V-x).$$

Здесь начало координат стоит в **вершине**, фокус поднят на высоту $f$, введено $Y = y - f$, лучи падают вдоль $y$. Это та же парабола, записанная в повёрнутых и сдвинутых осях — доказать это и есть задача В3 домашнего листа.

Стоит заметить, что **однородность уцелела в обеих записях**, хотя уравнения выглядят по-разному. Так и должно быть: масштабная симметрия — свойство задачи, а не системы координат. Важно только, относительно какой точки растягивать — относительно фокуса. Поэтому здесь и введено $Y = y - f$, а не просто $y$: в конспекте фокус уже стоит в начале координат, а тут его пришлось туда перенести.

---

Галочка **«семейство решений»** показывает другие решения того же уравнения, галочка **«поле направлений»** — почему оно однородное. Обе выключены по умолчанию: на занятии они не обязательны.


In [ ]:
# =============================================================
#  ФОРМА ЗЕРКАЛА — движок: профили, отражение, ОДУ формы.
#  Интерфейс — в следующей ячейке. Здесь ничего не рисуется.
# =============================================================
import numpy as np
from scipy.integrate import solve_ivp

# Масштаб выбран один раз и навсегда: радиус кривизны в вершине R = 2,
# значит параксиальный фокус на f = R/2 = 1. Всё остальное — в этих единицах;
# задача не содержит своей длины, поэтому выбор единицы ни на что не влияет.
R = 2.0
F = R / 2.0

SRC_CONIC, SRC_ODE = 0, 1


# ---------------- прямая задача: коника с параметром k ----------------
def conic_max_x(k):
    """Где коника кончается: под корнем R^2 - (1+k)x^2 должно быть >= 0."""
    c = 1.0 + k
    return np.inf if c <= 0.0 else R / np.sqrt(c)


def conic_profile(x, k):
    """Профиль конического сечения и его наклон.
        y = x^2 / (R + sqrt(R^2 - (1+k)x^2)),   y' = x / (R - (1+k)y)
    Это то же самое, что x^2 = 2Ry - (1+k)y^2, только без ветвления.
    k = 0 сфера, k = -1 парабола, -1<k<0 вытянутый эллипсоид,
    k > 0 сплюснутый эллипсоид, k < -1 гиперболоид."""
    x = np.asarray(x, dtype=float)
    s = np.sqrt(np.maximum(R ** 2 - (1.0 + k) * x ** 2, 0.0))
    y = x ** 2 / (R + s)
    return y, x / (R - (1.0 + k) * y)


# ---------------- обратная задача: ОДУ формы ----------------
def ode_slope(x, Y):
    """dY/dx для зеркала, собирающего вертикальные лучи в точку Y = 0.

    Закон отражения даёт квадратное уравнение на производную
        x y'^2 - 2Y y' - x = 0,   откуда   y' = (Y + sqrt(x^2 + Y^2)) / x.
    Здесь берётся та же формула, домноженная на сопряжённое:
        y' = x / (sqrt(x^2 + Y^2) - Y).
    Это не косметика. Под вершиной Y < 0, и в первом виде sqrt(x^2+Y^2) + Y —
    разность почти равных чисел: при x = 1e-6 теряется пять знаков. Во втором
    виде это сумма модулей, потерь нет вовсе.
    Вдобавок особенность на оси оказывается МНИМОЙ: при x = 0 первый вид даёт
    0/0, второй — честный ноль (и квадратное уравнение при x = 0 вырождается
    в -2Y y' = 0, то есть y' = 0 — ровно то, чего требует симметрия).
    Поэтому интегрировать можно прямо от вершины.

    Уравнение ОДНОРОДНОЕ: числитель и знаменатель первой степени, так что
    правая часть зависит только от Y/x. Это видно заранее, без выкладок: в
    условии «собрать все вертикальные лучи в одну точку» нет ни одной длины."""
    r = np.hypot(x, Y)
    d = r - Y
    return x / np.where(np.abs(d) < 1e-300, 1e-300, d)


def _rhs(x, Y):
    return [ode_slope(x, Y[0])]


X_END_ODE = 1.8        # интегрируем один раз до края самой широкой апертуры
_ODE_CACHE = {}


def _ode_solution(f):
    """Решение кэшируется по f: за кадр профиль нужен несколько раз
    (зеркало, лучи, график пересечений), а интегрировать хватит однажды."""
    key = round(float(f), 9)
    sol = _ODE_CACHE.get(key)
    if sol is None:
        if len(_ODE_CACHE) > 64:
            _ODE_CACHE.clear()
        sol = solve_ivp(_rhs, [0.0, X_END_ODE], [-f], dense_output=True,
                        rtol=1e-10, atol=1e-13)
        _ODE_CACHE[key] = sol
    return sol


def ode_profile(x, f=F):
    """Зеркало, построенное ИНТЕГРИРОВАНИЕМ уравнения, а не подстановкой
    готовой формулы: шаг за шагом из локального условия отражения, и слово
    «парабола» нигде не участвует.

    Начальное условие — вершина в начале координат, фокус на высоте f,
    то есть Y(0) = -f. Постоянная интегрирования и есть фокусное расстояние;
    само уравнение никакого масштаба не содержит.
    Зеркало чётно по x, поэтому считаем при |x| и переносим знак на наклон."""
    x = np.asarray(x, dtype=float)
    xa = np.abs(x)
    Y = _ode_solution(f).sol(np.clip(xa, 0.0, X_END_ODE))[0]
    return Y + f, np.sign(x) * ode_slope(xa, Y)


def exact_profile(x, f=F):
    """Точное решение того же ОДУ: Y = x^2/(2p) - p/2 при p = 2f.
    Подстановка Y = v x даёт x v' = sqrt(1+v^2), дальше разделение
    переменных: arsh v = ln x + C. Отсюда парабола с фокусным f."""
    p = 2.0 * f
    return np.asarray(x, dtype=float) ** 2 / (2.0 * p) - p / 2.0 + f


def profile(x, source, k=-1.0, f=F):
    """Единая точка входа: коника или решение ОДУ."""
    return conic_profile(x, k) if source == SRC_CONIC else ode_profile(x, f)


# ---------------- лучи ----------------
def trace(heights, source, k=-1.0, f=F):
    """Вертикальные лучи падают сверху, отражаются от зеркала.
    Возвращает точки попадания, направления и куда луч уходит."""
    y, yp = profile(heights, source, k, f)
    n = 1.0 / np.sqrt(1.0 + yp ** 2)
    nx, ny = -yp * n, n                  # единичная нормаль
    dot = -ny                            # падающий луч (0, -1), скалярно с n
    rx = -2.0 * dot * nx
    ry = -1.0 - 2.0 * dot * ny
    return y, rx, ry


def axis_crossing(heights, source, k=-1.0, f=F):
    """Высота, на которой отражённый луч пересекает ось x = 0.
    Это и есть «фокус для этого луча»: у идеального зеркала — одна и та же."""
    h = np.asarray(heights, dtype=float)
    y, rx, ry = trace(h, source, k, f)
    with np.errstate(divide='ignore', invalid='ignore'):
        t = np.where(np.abs(rx) > 1e-12, -h / rx, np.nan)
        out = np.where(t > 0, y + ry * t, np.nan)
    return out


def focal_spot(heights, source, k=-1.0, f=F):
    """Где луч протыкает фокальную плоскость y = f. Радиус пятна = разброс."""
    h = np.asarray(heights, dtype=float)
    y, rx, ry = trace(h, source, k, f)
    with np.errstate(divide='ignore', invalid='ignore'):
        t = np.where(np.abs(ry) > 1e-12, (f - y) / ry, np.nan)
        return np.where(t > 0, h + rx * t, np.nan)


def aberration(h_max, source, k=-1.0, f=F, n=41):
    """Сводка качества: продольная аберрация края и радиус пятна."""
    h = np.linspace(h_max / n, h_max, n)
    cr = axis_crossing(h, source, k, f)
    sp = focal_spot(h, source, k, f)
    cr_ok, sp_ok = cr[np.isfinite(cr)], sp[np.isfinite(sp)]
    return {
        'edge_focus': cr[-1] if np.isfinite(cr[-1]) else np.nan,
        'long_aber': abs(cr[-1] - f) if np.isfinite(cr[-1]) else np.nan,
        'focus_spread': (cr_ok.max() - cr_ok.min()) if cr_ok.size else np.nan,
        'spot_radius': np.abs(sp_ok).max() if sp_ok.size else np.nan,
    }


CONIC_NAMES = (
    (-1.0, 'парабола'),
    (0.0, 'сфера'),
)


def conic_name(k):
    if np.isclose(k, -1.0):
        return 'парабола'
    if np.isclose(k, 0.0):
        return 'сфера'
    if k > 0.0:
        return 'сплюснутый эллипсоид'
    if -1.0 < k < 0.0:
        return 'вытянутый эллипсоид'
    return 'гиперболоид'


print('движок загружен: R = %.0f, параксиальный фокус f = %.0f; '
      'ОДУ интегрируется прямо от вершины' % (R, F))

In [ ]:
# =============================================================
#  ФОРМА ЗЕРКАЛА — интерфейс.
#  Требует предыдущую ячейку (движок). Запускать после неё.
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

H_LIM = (0.10, 1.40)        # диапазон апертуры (полуширина пучка)
K_LIM = (-2.50, 1.00)       # диапазон параметра коники
CONTINUOUS = True           # False -- если на слабой машине ползунки дёргаются
N_SWEEP = 41                # точек в графике «где луч пересекает ось»
RAY_LEN = 5.0               # предел длины отражённого луча
RAY_PAST = 1.35             # во сколько раз продлить луч ЗА пересечение с осью
XLIM_PAD, YLIM = 0.25, (-0.70, 2.60)

_SL = dict(continuous_update=CONTINUOUS, style={'description_width': '120px'},
           layout=widgets.Layout(width='330px'))

_EQ_CONIC = (r"$y = \dfrac{x^{2}}{R + \sqrt{R^{2} - (1+k)\,x^{2}}}$" + "\n"
             r"$R = 2,\ \ f = R/2 = 1$")
_EQ_ODE = (r"$\dfrac{dY}{dx} = \dfrac{x}{\sqrt{x^{2}+Y^{2}} - Y}$" + "\n"
           r"$Y = y - f,\ \ Y(0) = -f$")


class MirrorApp:

    def __init__(self):
        self._busy = False
        self._sweep_key = None
        self._sweep = None
        self._legend_key = None
        self._build_controls()
        self._build_figure()
        self._wire()
        display(self.panel, self.out, self.info)
        self._refresh()

    # ---------------- органы управления ----------------
    def _build_controls(self):
        self.w_src = widgets.ToggleButtons(
            options=[('подобрать форму', SRC_CONIC), ('вывести форму', SRC_ODE)],
            value=SRC_CONIC, description='Задача:',
            style={'description_width': '70px', 'button_width': '160px'})
        self.w_k = widgets.FloatSlider(value=0.0, min=K_LIM[0], max=K_LIM[1],
                                       step=0.05, readout_format='.2f',
                                       description='форма k:', **_SL)
        self.w_h = widgets.FloatSlider(value=1.20, min=H_LIM[0], max=H_LIM[1],
                                       step=0.05, readout_format='.2f',
                                       description='апертура h:', **_SL)
        self.w_n = widgets.IntSlider(value=15, min=3, max=41, step=2,
                                     description='лучей:', **_SL)
        self.w_field = widgets.Checkbox(value=False, indent=False,
                                        description='поле направлений')
        self.w_family = widgets.Checkbox(value=False, indent=False,
                                         description='семейство решений')
        self.b_sphere = widgets.Button(description='Сфера (k = 0)',
                                       layout=widgets.Layout(width='150px'))
        self.b_parab = widgets.Button(description='Парабола (k = -1)',
                                      layout=widgets.Layout(width='170px'))
        self.b_reset = widgets.Button(description='Сброс',
                                      layout=widgets.Layout(width='90px'))
        self.out = widgets.Output()
        self.info = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Форма зеркала: "
                         "собрать все лучи в одну точку</h3>"),
            self.w_src,
            widgets.HBox([self.w_k, self.w_h, self.w_n]),
            widgets.HBox([self.b_sphere, self.b_parab, self.b_reset,
                          self.w_field, self.w_family]),
        ])

    def _wire(self):
        for w in (self.w_src, self.w_k, self.w_h, self.w_n,
                  self.w_field, self.w_family):
            w.observe(self._on_change, names='value')
        self.b_sphere.on_click(lambda _b: self._set(w_src=SRC_CONIC, w_k=0.0))
        self.b_parab.on_click(lambda _b: self._set(w_src=SRC_CONIC, w_k=-1.0))
        self.b_reset.on_click(lambda _b: self._set(
            w_src=SRC_CONIC, w_k=0.0, w_h=1.20, w_n=15))

    # ---------------- фигура строится ОДИН раз ----------------
    def _build_figure(self):
        self.fig, (self.ax, self.ax2) = plt.subplots(
            1, 2, figsize=(11.2, 4.9), dpi=88,
            gridspec_kw={'width_ratios': [2.6, 1.0]})
        self.fig.subplots_adjust(left=0.062, right=0.985, top=0.90,
                                 bottom=0.115, wspace=0.30)
        plt.close(self.fig)          # чтобы фигура не дублировалась под ячейкой

        ax = self.ax
        self.a_field = LineCollection([], colors='#a8a8a8', linewidths=1.0,
                                      zorder=1)
        ax.add_collection(self.a_field)
        # изоклины однородного уравнения — прямые через фокус; вдоль каждой
        # чёрточки поля параллельны, и это и есть определение однородности
        self.a_iso = LineCollection([], colors='#8c6fc4', linewidths=1.0,
                                    linestyles=':', alpha=0.75, zorder=1)
        ax.add_collection(self.a_iso)
        self.a_iso_mark = LineCollection([], colors='#6f4fb0', linewidths=2.0,
                                         zorder=3)
        ax.add_collection(self.a_iso_mark)
        self.a_family = LineCollection([], colors='#7fbf7f', linewidths=1.2,
                                       linestyles='--', zorder=2)
        ax.add_collection(self.a_family)
        ax.axvline(0.0, color='#999999', lw=1.0, ls='-.', zorder=2)
        self.a_inc = LineCollection([], colors='#ef8a00', linewidths=1.1,
                                    alpha=0.65, zorder=4)
        ax.add_collection(self.a_inc)
        self.a_ref = LineCollection([], colors='#1f4e9c', linewidths=1.1,
                                    alpha=0.9, zorder=5)
        ax.add_collection(self.a_ref)
        self.a_mirror, = ax.plot([], [], color='black', lw=3.0,
                                 label='зеркало', zorder=8)
        self.a_check, = ax.plot([], [], ls='none', marker='o', ms=5.0,
                                mfc='none', mec='#ff6a00', mew=1.5,
                                label='коника k = -1 (сверка)', zorder=9)
        self.a_focus, = ax.plot([0.0], [F], 'o', color='black', ms=8,
                                mfc='white', mew=1.6,
                                label='параксиальный фокус', zorder=10)
        self.t_eq = ax.text(0.015, 0.035, '', transform=ax.transAxes,
                            fontsize=12, va='bottom', ha='left',
                            bbox=dict(fc='white', ec='#cccccc', alpha=0.92,
                                      boxstyle='round,pad=0.35'))
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_ylim(*YLIM)
        ax.grid(True, ls='--', alpha=0.35)
        ax.set_adjustable('box')

        a2 = self.ax2
        self.a_par_line = a2.axhline(F, color='#999999', lw=1.2, ls='--',
                                     label='параксиальный фокус')
        self.a_cross, = a2.plot([], [], color='#1f4e9c', lw=2.2,
                                label='куда попал луч')
        a2.set_xlabel('высота луча h')
        a2.set_ylabel('где пересёк ось')
        a2.grid(True, ls='--', alpha=0.35)
        a2.set_title('идеально = горизонтальная прямая', fontsize=9.5)
        a2.legend(fontsize=7.5, loc='lower left')

    # ---------------- график пересечений (с кэшем) ----------------
    def _sweep_curve(self, src, k, h):
        key = (src, k, h)
        if key == self._sweep_key:
            return self._sweep
        hs = np.linspace(h / N_SWEEP, h, N_SWEEP)
        self._sweep_key = key
        self._sweep = (hs, axis_crossing(hs, src, k))
        return self._sweep

    # ---------------- поле направлений ----------------
    def _field_segments(self, x_hi):
        X, Y = np.meshgrid(np.linspace(-x_hi, x_hi, 19),
                           np.linspace(YLIM[0] + 0.15, YLIM[1] - 0.15, 13))
        S = ode_slope(X, Y - F)
        n = (0.055 * x_hi) / np.hypot(1.0, S)
        return np.stack([np.stack([X - n, Y - n * S], axis=-1),
                         np.stack([X + n, Y + n * S], axis=-1)],
                        axis=-2).reshape(-1, 2, 2)

    ISO_V = (-1.5, -0.5, 0.5, 1.5)

    def _isocline_segments(self, x_hi):
        """Прямые Y = v·x через фокус плюс сами чёрточки поля на них.

        В этом и состоит однородность: правая часть зависит только от Y/x,
        поэтому вдоль каждой такой прямой наклон один и тот же — фиолетовые
        чёрточки на одной прямой параллельны, как их ни двигай от фокуса."""
        lines = [[(-x_hi, F - v * x_hi), (x_hi, F + v * x_hi)]
                 for v in self.ISO_V]
        marks = []
        L = 0.06 * x_hi
        for v in self.ISO_V:
            s = ode_slope(1.0, v)                 # наклон вдоль всей прямой
            n = L / np.hypot(1.0, s)
            for x in np.linspace(0.22 * x_hi, 0.95 * x_hi, 4):
                for sx in (x, -x):
                    y = F + v * sx
                    if YLIM[0] < y < YLIM[1]:
                        ss = s if sx > 0 else -s
                        nn = L / np.hypot(1.0, ss)
                        marks.append([(sx - nn, y - nn * ss),
                                      (sx + nn, y + nn * ss)])
        return lines, marks

    # ---------------- отрисовка ----------------
    def render(self):
        src, k = self.w_src.value, self.w_k.value
        h, n_rays = self.w_h.value, self.w_n.value
        self.w_k.disabled = (src == SRC_ODE)

        # коника существует не при любой апертуре -- обрезаем честно
        h_lim = conic_max_x(k) if src == SRC_CONIC else np.inf
        clipped = h > 0.98 * h_lim
        h_eff = min(h, 0.98 * h_lim)

        xm = np.linspace(-h_eff, h_eff, 401)
        y_mirror, _ = profile(xm, src, k)
        self.a_mirror.set_data(xm, y_mirror)
        handles = [self.a_mirror]

        # в режиме вывода показываем, что получилось ровно k = -1
        if src == SRC_ODE:
            xc = np.linspace(-h_eff, h_eff, 17)
            self.a_check.set_data(xc, conic_profile(xc, -1.0)[0])
            handles.append(self.a_check)
            dev = np.max(np.abs(y_mirror - conic_profile(xm, -1.0)[0]))
        else:
            self.a_check.set_data([], [])
            dev = None

        # лучи
        hs = np.linspace(-h_eff, h_eff, n_rays)
        y_hit, rx, ry = trace(hs, src, k)
        top = YLIM[1]
        self.a_inc.set_segments([[(xi, top), (xi, yi)]
                                 for xi, yi in zip(hs, y_hit)])
        # луч ведём чуть дальше его собственного пересечения с осью: так виден
        # и сход, и расход, но без звёздного месива до краёв рамки
        with np.errstate(divide='ignore', invalid='ignore'):
            t_ax = np.where(np.abs(rx) > 1e-9, -hs / rx, np.nan)
        t_end = np.where(np.isfinite(t_ax) & (t_ax > 0),
                         RAY_PAST * t_ax, RAY_LEN)
        t_end = np.minimum(t_end, RAY_LEN)
        self.a_ref.set_segments([[(xi, yi), (xi + rxi * ti, yi + ryi * ti)]
                                 for xi, yi, rxi, ryi, ti
                                 in zip(hs, y_hit, rx, ry, t_end)])

        # поле направлений и семейство решений
        x_hi = max(1.0, h_eff) + XLIM_PAD
        if self.w_field.value:
            iso_lines, iso_marks = self._isocline_segments(x_hi)
            self.a_field.set_segments(self._field_segments(x_hi))
            self.a_iso.set_segments(iso_lines)
            self.a_iso_mark.set_segments(iso_marks)
        else:
            self.a_field.set_segments([])
            self.a_iso.set_segments([])
            self.a_iso_mark.set_segments([])
        if self.w_family.value:
            xf = np.linspace(-x_hi, x_hi, 201)
            self.a_family.set_segments(
                [np.column_stack([xf, ode_profile(xf, f=p / 2.0)[0]])
                 for p in (1.0, 3.0)])
        else:
            self.a_family.set_segments([])

        # когда включено поле, лучи уходят на второй план: смотрим на поле
        self.a_ref.set_alpha(0.30 if self.w_field.value else 0.90)
        self.a_inc.set_alpha(0.22 if self.w_field.value else 0.65)

        self.ax.set_xlim(-x_hi, x_hi)
        self.t_eq.set_text(_EQ_CONIC if src == SRC_CONIC else _EQ_ODE)
        handles.append(self.a_focus)

        key = tuple(id(a) for a in handles)
        if key != self._legend_key:
            self.ax.legend(handles=handles, loc='upper right', fontsize=8.5)
            self._legend_key = key

        ab = aberration(h_eff, src, k)
        perfect = not np.isfinite(ab['focus_spread']) or ab['focus_spread'] < 1e-6
        if src == SRC_ODE:
            ttl, col = ('Форма получена из уравнения — все лучи в одной точке',
                        'green')
        elif perfect:
            ttl, col = ('%s: все лучи собираются в одну точку'
                        % conic_name(k).capitalize(), 'green')
        elif not np.isfinite(ab['edge_focus']):
            ttl, col = ('%s: краевой луч вообще не возвращается на ось'
                        % conic_name(k).capitalize(), '#b00000')
        else:
            ttl, col = ('%s: край собирается на %.3f вместо %.3f — '
                        'промах %.1f%% от f'
                        % (conic_name(k).capitalize(), ab['edge_focus'], F,
                           ab['long_aber'] / F * 100), '#b00000')
        self.ax.set_title(ttl, fontsize=12.5, color=col, pad=8)

        hs_s, cr_s = self._sweep_curve(src, k, h_eff)
        self.a_cross.set_data(hs_s, cr_s)
        fin = cr_s[np.isfinite(cr_s)]
        lo, hi = (F - 0.1, F + 0.1) if fin.size == 0 else (fin.min(), fin.max())
        pad = max(0.08, 0.15 * (hi - lo))
        self.ax2.set_xlim(0.0, h_eff)
        self.ax2.set_ylim(min(lo, F) - pad, max(hi, F) + pad)

        return self._info_html(src, k, h, h_eff, clipped, ab, dev)

    # ---------------- числа под графиком ----------------
    def _info_html(self, src, k, h, h_eff, clipped, ab, dev):
        def good(ok, txt):
            return "<b style='color:%s'>%s</b>" % ('green' if ok else '#b00000', txt)

        f_num = F / (2.0 * h_eff)
        rows = [
            "продольная аберрация: %s <span style='color:#777'>(край на %s, "
            "фокус %.3f)</span>"
            % (good(ab['long_aber'] < 1e-6,
                    '%.4f = %.1f%% от f' % (ab['long_aber'],
                                            ab['long_aber'] / F * 100)
                    if np.isfinite(ab['long_aber']) else 'луч не вернулся'),
               '%.3f' % ab['edge_focus'] if np.isfinite(ab['edge_focus']) else '—',
               F),
            "радиус пятна в фокальной плоскости: %s"
            % good(ab['spot_radius'] < 1e-6,
                   '%.4f' % ab['spot_radius']
                   if np.isfinite(ab['spot_radius']) else '—'),
            "светосила: f/%.2f" % f_num,
        ]
        if src == SRC_CONIC:
            head = ('форма задана как коника: k = %.2f — <b>%s</b>'
                    % (k, conic_name(k)))
            tail = (
                "Аберрация растёт как <b>h²</b> (пятно — как h³): сузьте "
                "апертуру вдвое, и промах упадёт вчетверо. Поэтому фокус и "
                "называется <i>параксиальным</i> — сфера безупречна в пределе "
                "узкого пучка, а «парабола лучше сферы» — это утверждение "
                "про широкие пучки.")
        else:
            head = ('форма НЕ задана, а получена: уравнение проинтегрировано '
                    'от вершины наружу, шаг за шагом из условия отражения')
            tail = (
                "Совпадение с коникой k = -1: расхождение <b>%.1e</b> на всей "
                "апертуре. Уравнение однородное — правая часть зависит только "
                "от Y/x. Это видно до вычислений: в условии «собрать все "
                "вертикальные лучи в одну точку» нет ни одной длины, значит "
                "растяжение (x, Y) → (λx, λY) обязано переводить решения в "
                "решения. Подстановка Y = vx даёт x·v′ = √(1+v²), дальше "
                "разделение переменных." % (dev if dev is not None else np.nan))
        note = ''
        if clipped:
            note = ("<div style='color:#b06000;margin-top:4px'>При k = %.2f "
                    "коника существует только до |x| = %.3f, поэтому апертура "
                    "обрезана с %.2f до %.2f.</div>" % (k, conic_max_x(k), h, h_eff))
        if self.w_field.value:
            note += ("<div style='color:#6f4fb0;margin-top:4px'>Серые чёрточки "
                     "— поле направлений уравнения. Фиолетовые пунктиры — "
                     "прямые Y = v·x через фокус: вдоль каждой наклон "
                     "постоянен (жирные чёрточки на ней параллельны). Это и "
                     "есть однородность — правая часть зависит только от "
                     "отношения Y/x, а не от x и Y по отдельности. Зеркало — "
                     "одна из интегральных кривых этого поля.</div>")
        if self.w_family.value:
            note += ("<div style='color:#3a7a3a;margin-top:4px'>Зелёные "
                     "пунктиры — другие решения того же уравнения. Все они "
                     "получаются друг из друга растяжением относительно "
                     "фокуса: масштаб сидит не в уравнении, а в начальном "
                     "условии.</div>")
        return ("<div style='font-size:13px;line-height:1.65;max-width:1120px'>"
                "<div>%s</div><div>%s</div><div style='margin-top:3px'>%s</div>"
                "%s</div>"
                % (head, ' &nbsp;•&nbsp; '.join(rows), tail, note))

    # ---------------- реакция на события ----------------
    def _refresh(self):
        self.info.value = self.render()
        with self.out:
            clear_output(wait=True)
            display(self.fig)

    def _on_change(self, change=None):
        if not self._busy:
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()


app = MirrorApp()

---

## Где остановиться: аберрация против дифракции

Из сценария выше следует вывод: чем уже пучок, тем меньше пятно, значит узкая апертура всегда лучше. Это вывод геометрической оптики, и он неверен.

Свет — волна. Даже безупречное зеркало собирает параллельный пучок не в точку, а в кружок Эйри радиусом

$$\rho_{\text{дифр}} \approx 1{,}22\,\frac{\lambda f}{D}, \qquad D = 2h .$$

Этот вклад при сужении пучка **растёт**. Геометрический падает как $h^3$, дифракционный растёт как $1/h$ — значит у суммы есть минимум, и он говорит, какое зеркало вообще имеет смысл делать.

Ячейка ниже считает оба вклада: геометрический — тем же трассировщиком, что и весь виджет (а не по подогнанной формуле), дифракционный — по формуле выше.


In [ ]:
# =============================================================
#  АБЕРРАЦИЯ ПРОТИВ ДИФРАКЦИИ: где остановиться с апертурой.
#  Требует ячейку с движком: геометрия берётся тем же трассировщиком,
#  что и весь виджет, а не по подогнанной формуле.
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

LAM = 550e-9                      # длина волны, м (зелёный свет)


def spot_geom_rel(x):
    """Радиус геометрического пятна СФЕРЫ в долях f; x = h/f."""
    return aberration(x, SRC_CONIC, k=0.0)['spot_radius'] / F


def spot_total(x, f_m, lam=LAM):
    """Полный радиус пятна в метрах.
    Геометрия масштабируется вместе с зеркалом: rho_g = f * g(h/f).
    Дифракция rho_d = 1.22 lam f / D, где D = 2h = 2 x f, то есть
    rho_d = 1.22 lam / (2x) -- от фокусного расстояния НЕ зависит."""
    return spot_geom_rel(x) * f_m + 1.22 * lam / (2.0 * x)


def optimum(f_m, lam=LAM):
    r = minimize_scalar(lambda t: spot_total(t, f_m, lam),
                        bounds=(0.004, 0.6), method='bounded',
                        options={'xatol': 1e-7})
    x = r.x
    return dict(x=x, D=2 * x * f_m, fnum=1.0 / (2 * x), spot=r.fun,
                geom=spot_geom_rel(x) * f_m, diff=1.22 * lam / (2 * x))


# ---- заодно проверим, что геометрическое пятно и правда ~ h^3 ----
_h = np.array([0.05, 0.10, 0.20, 0.40])
_slope = np.polyfit(np.log(_h), np.log([spot_geom_rel(t) for t in _h]), 1)[0]
print('показатель степени геометрического пятна (наклон в лог-лог): %.3f' % _slope)

# ---- график для f = 1 м ----
F_M = 1.0
x = np.logspace(np.log10(0.004), np.log10(0.6), 400)
g = np.array([spot_geom_rel(t) for t in x]) * F_M
d = 1.22 * LAM / (2 * x)
opt = optimum(F_M)

fig, ax = plt.subplots(figsize=(7.4, 4.5))
ax.loglog(x, g * 1e6, '--', lw=1.6, color='C0', label=r'геометрия (сфера), $\sim h^3$')
ax.loglog(x, d * 1e6, ':', lw=1.8, color='C2', label=r'дифракция, $\sim 1/h$')
ax.loglog(x, (g + d) * 1e6, '-', lw=2.2, color='C3', label='сумма')
ax.plot(opt['x'], opt['spot'] * 1e6, 'o', color='C3', ms=8, zorder=5)
ax.annotate('оптимум\n$h/f$ = %.3f,  $D$ = %.0f мм\nсветосила f/%.1f,  пятно %.1f мкм'
            % (opt['x'], opt['D'] * 1000, opt['fnum'], opt['spot'] * 1e6),
            xy=(opt['x'], opt['spot'] * 1e6), xytext=(18, 18),
            textcoords='offset points', fontsize=9,
            arrowprops=dict(arrowstyle='->', lw=0.8))
ax.set_xlabel(r'$h/f$  —  полуширина пучка в долях фокусного расстояния')
ax.set_ylabel('радиус пятна, мкм')
ax.set_title(r'Сферическое зеркало, $f$ = %g м, $\lambda$ = %g нм' % (F_M, LAM * 1e9))
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper center')
plt.tight_layout()
plt.show()

# ---- таблица по фокусным расстояниям ----
print()
print('   f, м |  h/f опт | D, мм | светосила | пятно, мкм   (геом + дифр)')
print('  ' + '-' * 66)
for f_m in (0.25, 0.5, 1.0, 2.0, 10.0):
    o = optimum(f_m)
    print('  %6g | %8.4f | %5.0f |  f/%-6.1f | %6.1f       (%.1f + %.1f)'
          % (f_m, o['x'], o['D'] * 1000, o['fnum'], o['spot'] * 1e6,
             o['geom'] * 1e6, o['diff'] * 1e6))

print()
print('  проверка степенного закона  x_опт ~ (lam/f)^(1/4):')
for f_m in (0.25, 1.0, 10.0):
    print('     f = %5g м:  трассировка %.5f,   оценка (1.525 lam/f)^(1/4) = %.5f'
          % (f_m, optimum(f_m)['x'], (1.525 * LAM / f_m) ** 0.25))

print()
print('  для сравнения ПАРАБОЛА при f = 1 м (геометрического пятна нет вовсе,')
print('  остаётся одна дифракция):')
for D_mm in (61, 200, 500):
    print('     D = %3d мм  ->  радиус пятна %.1f мкм' % (D_mm, 1.22 * LAM * 1.0 / (D_mm / 1000) * 1e6))


### Что из этого следует

**Сфера ограничивает апертуру.** При $f = 1$ м лучшее, что можно выжать из сферического зеркала, — пятно около 15 мкм, и достигается оно при диаметре всего 61 мм. Делать шире бессмысленно: аберрация съест выигрыш быстрее, чем дифракция его даст.

**Чем больше зеркало, тем хуже.** Оптимальная светосила растёт как $f^{1/4}$: f/11 при $f = 25$ см, f/16 при метре, f/29 при десяти метрах. Большое сферическое зеркало приходится делать всё более длиннофокусным — а это буквально длина трубы телескопа.

**Парабола снимает ограничение целиком.** Геометрического пятна у неё нет вовсе, остаётся одна дифракция: при $f = 1$ м и $D = 200$ мм это 3,4 мкм — вчетверо лучше предела сферы, и при втрое большей апертуре, то есть с вдесятеро большим потоком света.

Вот почему сферические зеркала спокойно жили столетиями (при f/10 и скромном диаметре они честно работают) и почему всё крупное всё-таки делают параболическим.

> **Оговорка.** Складывать радиусы двух пятен — оценка, а не расчёт: на самом деле складываются распределения интенсивности, и честная величина — диаметр, в котором собрано, скажем, 80 % энергии. Для вопроса «где примерно оптимум» этого хватает, порядок величины верен. Полезная привычка: заметить границу применимости собственной оценки сразу, а не после того, как её опровергнут.


---

## Вопросы к виджету

1. **Можно ли было сказать, что уравнение однородное, не выводя его?** Что именно в формулировке задачи это гарантирует?

<details><summary>Ответ</summary>

Можно. В условии «собрать все вертикальные лучи в одну точку» не участвует **ни одна длина**: ни размер зеркала, ни фокусное расстояние, ни длина волны. Значит если какая-то форма решает задачу, то и любая её гомотетия относительно фокуса тоже решает: растянули всё в $\lambda$ раз — углы не изменились, отражение осталось отражением.

А уравнение, множество решений которого переходит в себя при $(x, Y) \to (\lambda x, \lambda Y)$, обязано зависеть только от отношения $Y/x$. Фокусное расстояние берётся не из уравнения, а из начального условия.

</details>

2. **Сузьте апертуру ровно вдвое** (например, с 1,20 до 0,60) и посмотрите на продольную аберрацию. Во сколько раз она упала? Совпало ли с $h^2$?

<details><summary>Ответ</summary>

Примерно вчетверо, и это закон $h^2$ — но выполняется он тем точнее, чем уже пучок (при $h \sim 1$ мы далеко за пределами приближения). Числа трассировщика для сферы:

| $h$ | промах вдоль оси | радиус пятна |
|---|---|---|
| 1,20 | 25,0 % от $f$ | 85,7 % |
| 0,60 | 4,83 % | 3,37 % |
| 0,30 | 1,14 % | 0,355 % |
| 0,15 | 0,28 % | 0,043 % |

Отношения соседних промахов: 5,2 и 4,2 — сходятся к 4. Отношения пятен: 25 и 9,5 — сходятся к 8. Значит продольная аберрация растёт как $h^2$, поперечная как $h^3$. Прямая проверка наклона в лог-лог на малых $h$ даёт показатель 3,04.

</details>

3. **Почему параксиальный фокус называется параксиальным?** Сфера ведь «неправильная» — в каком смысле она всё-таки правильная?

<details><summary>Ответ</summary>

«Параксиальный» = «околоосевой». Сфера безупречна **в пределе** бесконечно узкого пучка: промах стремится к нулю как $h^2$, поэтому предельная точка сбора существует и равна $R/2$. Её и называют фокусом.

Сфера «неправильна» не вообще, а начиная с четвёртого порядка. Разложите обе кривые у вершины:

$$\text{сфера: } R - \sqrt{R^2-x^2} = \frac{x^2}{2R} + \frac{x^4}{8R^3} + \ldots, \qquad \text{парабола: } \frac{x^2}{2R}.$$

Совпадают вершина и кривизна в ней, расходится следующий член. Поэтому у сферы и параболы с одинаковым $R$ **один и тот же** параксиальный фокус — и поэтому же сфера годится тем лучше, чем уже пучок.

</details>

4. **Включите «семейство решений».** Чем отличаются зелёные кривые друг от друга? Какое преобразование переводит одну в другую и почему оно обязано сохранять множество решений?

<details><summary>Ответ</summary>

Зелёные кривые — параболы с разными фокусными расстояниями, то есть одна и та же кривая, растянутая относительно фокуса. Переводит одну в другую гомотетия $(x, Y) \to (\lambda x, \lambda Y)$ с центром в фокусе.

Сохранять множество решений она обязана по той же причине, что и в вопросе 1: у задачи нет своей длины. Это буквально одна и та же симметрия — в вопросе 1 она видна в формуле (правая часть зависит только от $Y/x$), а здесь на картинке.

</details>

5. **Откуда в ответе логарифм?** (Подсказка: посмотрите, в какой момент выкладки он появляется, и какое свойство задачи за это отвечает.)

<details><summary>Ответ</summary>

Логарифм приходит из разделения переменных. После подстановки $Y = vx$ остаётся

$$\frac{dv}{\sqrt{1+v^2}} = \frac{dx}{x},$$

и правая часть даёт $\ln x$. Она даст его в **любой** однородной задаче: однородность ровно и означает, что $x$ входит только через $dx/x$.

Смысл: при растяжении $x \to \lambda x$ логарифм меняется на константу, а константу всегда можно спрятать в постоянную интегрирования. Функция, которая не замечает масштаб с точностью до сдвига, — это и есть логарифм.

</details>

6. **Особая точка.** На оси $x = 0$ первая форма правой части даёт $0/0$. После домножения на сопряжённое особенность исчезает. Куда она делась — была ли она свойством *задачи* или свойством *записи*? Что говорит исходное квадратное уравнение при $x = 0$?

<details><summary>Ответ</summary>

Свойством **записи**. Исходное квадратное уравнение $x\,y'^2 - 2Y y' - x = 0$ при $x = 0$ вырождается в $-2Y y' = 0$, и раз в вершине $Y = -f \neq 0$, остаётся $y' = 0$ — ровно то, чего требует симметрия: вершина зеркала горизонтальна. Никакой неопределённости у задачи нет.

$0/0$ возникает потому, что у квадратного уравнения две ветви, и вторая при $x \to 0$ уходит в бесконечность; форма $y' = (Y + \sqrt{x^2+Y^2})/x$ делит на $x$ и вытаскивает эту особенность наружу. Домножение на сопряжённое просто выбирает нужную ветвь явно.

Практический вывод, который дороже самой задачи: если численный метод спотыкается на границе области, сначала проверьте, не спотыкается ли он о **форму записи**, и только потом ищите физику.

</details>

7. **Перейдите через $k = -1$** (например, поставьте $k = -1{,}5$, гиперболоид). Край собирается **выше** фокуса, а не ниже. Почему знак промаха меняется именно на параболе?

<details><summary>Ответ</summary>

Потому что парабола — не «лучшая из коник», а **точное** решение: у неё промах равен нулю при любой апертуре. Промах непрерывно зависит от $k$ и при $k = -1$ обращается в тождественный ноль — значит по разные стороны от $-1$ он разных знаков.

Трассировщик при $h = 0{,}8$ (фокус на 1):

| $k$ | −2,0 | −1,5 | −1,0 | −0,5 | 0,0 |
|---|---|---|---|---|---|
| промах | +0,083 | +0,042 | 0 | −0,044 | −0,091 |

Почти линейно по $(1+k)$ и с чистым нулём в параболе. Сфера недоворачивает края — они пересекают ось ближе к зеркалу; гиперболоид переворачивает знак.

</details>

## Жёлтая рамка (необязательное, на дом)

Оцените по закону $h^3$, при какой апертуре радиус пятна сферического зеркала станет меньше 1 % от фокусного расстояния. Отправная точка: при $h = 0{,}30$ пятно равно 0,36 % от $f$. Полученное значение проверьте ползунком.

<details>
<summary>Ответ</summary>

Из $h^3$: нужно увеличить пятно в $1/0{,}36 \approx 2{,}8$ раза, значит апертуру — в $\sqrt[3]{2{,}8} \approx 1{,}41$ раза, то есть примерно до $h \approx 0{,}42$. Ползунок даёт: при $h = 0{,}40$ пятно 0,88 %, при $h = 0{,}45$ уже 1,28 % — порог около 0,41. Оценка по кубическому закону сошлась.

В переводе на светосилу это $f/1{,}2$ — всё ещё очень «быстрое» зеркало. Значение по умолчанию $h = 1{,}20$ при $f = 1$ отвечает светосиле $f/0{,}42$, каких зеркал не бывает вовсе: взято, чтобы аберрация была видна на глаз. У настоящих телескопов $f/4 \ldots f/10$, и там сферическое зеркало ведёт себя вполне прилично — именно поэтому им долго и пользовались.
</details>

---

*Что можно править в коде:* геометрия и масштаб (`R`, отсюда `F = R/2`) — в начале ячейки с движком; диапазоны ползунков (`H_LIM`, `K_LIM`), число лучей в правой панели (`N_SWEEP`) и переключатель `CONTINUOUS` (плавное обновление при протяжке; поставьте `False`, если на слабой машине картинка дёргается) — в начале ячейки с интерфейсом.

> К жёлтой рамке есть продолжение: см. раздел **«Где остановиться: аберрация против дифракции»** выше. Если учесть волновую природу света, у пятна появляется минимум, и сужать апертуру бесконечно нельзя — а значит у сферического зеркала есть предел качества, которого парабола не имеет.
